# Random Forest Project 

For this project we will be exploring publicly available data from [LendingClub.com](https://www.lendingclub.com/). Lending Club connects people who need money (borrowers) with people who have money (investors). Hopefully, as an investor you would want to invest in people who showed a profile of having a high probability of paying you back. We will try to create a model that will help predict this.

Lending club had a [very interesting year in 2016](https://en.wikipedia.org/wiki/Lending_Club#2016), so let's check out some of their data and keep the context in mind. This data is from before they even went public.

We will use lending data from 2007-2010 and be trying to classify and predict whether or not the borrower paid back their loan in full. You can use the csv already provided on Blackboard. The csv provided has been cleaned of NA values.

Here are what the columns represent:
* credit.policy: 1 if the customer meets the credit underwriting criteria of LendingClub.com, and 0 otherwise.
* purpose: The purpose of the loan (takes values "credit_card", "debt_consolidation", "educational", "major_purchase", "small_business", and "all_other").
* int.rate: The interest rate of the loan, as a proportion (a rate of 11% would be stored as 0.11). Borrowers judged by LendingClub.com to be more risky are assigned higher interest rates.
* installment: The monthly installments owed by the borrower if the loan is funded.
* log.annual.inc: The natural log of the self-reported annual income of the borrower.
* dti: The debt-to-income ratio of the borrower (amount of debt divided by annual income).
* fico: The FICO credit score of the borrower.
* days.with.cr.line: The number of days the borrower has had a credit line.
* revol.bal: The borrower's revolving balance (amount unpaid at the end of the credit card billing cycle).
* revol.util: The borrower's revolving line utilization rate (the amount of the credit line used relative to total credit available).
* inq.last.6mths: The borrower's number of inquiries by creditors in the last 6 months.
* delinq.2yrs: The number of times the borrower had been 30+ days past due on a payment in the past 2 years.
* pub.rec: The borrower's number of derogatory public records (bankruptcy filings, tax liens, or judgments).

**Name:** Alghadeer Hussain Alwalah  
**ID:** 2240007327  
**Section:** 6FS4

# Import Libraries

**Import the usual libraries for pandas and plotting. You can import sklearn later on.**

In [ ]:
# importing pandas for data manipulation and analysis
import pandas as pd

# importing numpy for numerical operations
import numpy as np

# importing matplotlib for basic plotting
import matplotlib.pyplot as plt

# importing seaborn for better visualizations
import seaborn as sns

%matplotlib inline

## Get the Data

** Use pandas to read loan_data.csv as a dataframe called loans.**

In [ ]:
# reading the loan dataset into a dataframe
loans = pd.read_csv('loan_data.csv')

** Check out the info(), head(), and describe() methods on loans.**

In [ ]:
# checking the structure and data types of the dataset
loans.info()

In [ ]:
# previewing the first 5 rows to understand the data
loans.head()

In [ ]:
# getting statistical summary of the dataset
loans.describe()

# Exploratory Data Analysis

**Create a histogram of two FICO distributions on top of each other, one for each credit.policy outcome.**

In [ ]:
# plotting FICO score distribution based on credit policy
# blue = customers who meet credit policy, red = those who don't
plt.figure(figsize=(10,6))
loans[loans['credit.policy']==1]['fico'].hist(alpha=0.5, color='blue', bins=30, label='Credit Policy=1')
loans[loans['credit.policy']==0]['fico'].hist(alpha=0.5, color='red', bins=30, label='Credit Policy=0')
plt.legend()
plt.xlabel('FICO')
plt.title('FICO Distribution by Credit Policy')
plt.show()

**Create a similar figure, except this time select by the not.fully.paid column.**

In [ ]:
# plotting FICO score distribution based on whether the loan was fully paid
# blue = not fully paid, red = fully paid
plt.figure(figsize=(10,6))
loans[loans['not.fully.paid']==1]['fico'].hist(alpha=0.5, color='blue', bins=30, label='not.fully.paid=1')
loans[loans['not.fully.paid']==0]['fico'].hist(alpha=0.5, color='red', bins=30, label='not.fully.paid=0')
plt.legend()
plt.xlabel('FICO')
plt.title('FICO Distribution by Not Fully Paid')
plt.show()

**Create a countplot using seaborn showing the counts of loans by purpose, with the color hue defined by not.fully.paid.**

In [ ]:
# countplot to see how many loans were not fully paid for each loan purpose
plt.figure(figsize=(11,7))
sns.countplot(x='purpose', hue='not.fully.paid', data=loans, palette='Set1')
plt.title('Loan Purpose Count by Not Fully Paid')
plt.show()

**Let's see the trend between FICO score and interest rate. Create a jointplot using seaborn.**

In [ ]:
# jointplot to see the relationship between FICO score and interest rate
# higher FICO score usually means lower interest rate
sns.jointplot(x='fico', y='int.rate', data=loans, color='purple')
plt.show()

**Create the following lmplots to see if the trend differed between not.fully.paid and credit.policy.**

In [ ]:
# lmplot to compare FICO vs interest rate trends
# separated by credit policy and whether the loan was fully paid
plt.figure(figsize=(11,7))
sns.lmplot(x='fico', y='int.rate', data=loans, hue='credit.policy',
           col='not.fully.paid', palette='Set1')
plt.show()

# Setting up the Data

**Create a list of 1 element containing the string 'purpose'. Call this list cat_feats.**

In [ ]:
# storing the categorical feature in a list to use with get_dummies
cat_feats = ['purpose']

**Now use pd.get_dummies(loans, columns=cat_feats, drop_first=True) to create a fixed larger dataframe that has new feature columns with dummy variables. Set this dataframe as final_data.**

In [ ]:
# converting the categorical 'purpose' column into dummy/indicator variables
# drop_first=True to avoid multicollinearity
final_data = pd.get_dummies(loans, columns=cat_feats, drop_first=True)

## Train Test Split

**Use sklearn to split your data into a training set and a testing set.**

In [ ]:
# importing train_test_split to split data into training and testing sets
from sklearn.model_selection import train_test_split

In [ ]:
# defining features (X) and target variable (y)
X = final_data.drop('not.fully.paid', axis=1)
y = final_data['not.fully.paid']

# splitting data: 70% training, 30% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=101)

## Training a Decision Tree Model

**Let's start by training a single decision tree first!**

**Create an instance of DecisionTreeClassifier() called dtree and fit it to the training data.**

In [ ]:
# importing DecisionTreeClassifier from sklearn
from sklearn.tree import DecisionTreeClassifier

In [ ]:
# creating a Decision Tree model instance
dtree = DecisionTreeClassifier()

# training the model on the training data
dtree.fit(X_train, y_train)

## Predictions and Evaluation of Decision Tree
**Create predictions from the test set and create a classification report and a confusion matrix.**

In [ ]:
# importing evaluation metrics
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# using the trained decision tree to make predictions on test data
predictions = dtree.predict(X_test)

In [ ]:
# printing classification report to evaluate precision, recall, and f1-score
print(classification_report(y_test, predictions))

In [ ]:
# printing confusion matrix to see actual vs predicted values
print(confusion_matrix(y_test, predictions))

## Training the Random Forest model

Now its time to train our model!

**Create an instance of the RandomForestClassifier class and fit it to our training data from the previous step.**

In [ ]:
# importing RandomForestClassifier from sklearn
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# creating a Random Forest model with 600 trees for better accuracy
rfc = RandomForestClassifier(n_estimators=600)

# training the random forest model on the training data
rfc.fit(X_train, y_train)

## Predictions and Evaluation

Let's predict off the y_test values and evaluate our model.

** Predict the class of not.fully.paid for the X_test data.**

In [ ]:
# using the trained random forest model to predict on test data
rfc_pred = rfc.predict(X_test)

**Now create a classification report from the results. Do you get anything strange or some sort of warning?**

In [ ]:
# classification report for random forest - notice class 1 recall is very low
# this happens because the dataset is imbalanced (more fully paid loans)
print(classification_report(y_test, rfc_pred))

In [ ]:
# confusion matrix for random forest predictions
print(confusion_matrix(y_test, rfc_pred))

**What performed better the random forest or the decision tree?**

# Great Job!